# 07a — Tutorial 3: Solvation Free Energy of CO₂ vs. Bicarbonate via Free Energy Perturbation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/Module2_MolecularDynamics/notebooks/07a_tutorial_co2_bicarbonate_fep.ipynb)

## 🎯 Learning Objectives

- Understand the thermodynamic cycle connecting CO₂ and bicarbonate (HCO₃⁻) solvation
- Derive the Free Energy Perturbation (FEP) formula from first principles
- Implement the Zwanzig exponential averaging estimator and its limitations
- Apply Bennett Acceptance Ratio (BAR) for improved free energy estimates
- Construct and analyze the alchemical λ-schedule for a charge-change perturbation
- Compute and interpret the free energy of hydration difference $\Delta\Delta G_{\text{hyd}}$
- Understand the Born model and why ionic species are far more hydrated than neutral ones
- Perform a full staged FEP calculation entirely in Python (no external MD code required)

---
## 1. Scientific Background

### 1.1 The CO₂ Hydration Equilibrium

Carbon dioxide dissolves in water and can undergo hydration and ionization:

$$\text{CO}_2(\text{aq}) + \text{H}_2\text{O} \rightleftharpoons \text{H}_2\text{CO}_3 \rightleftharpoons \text{H}^+ + \text{HCO}_3^-$$

The thermodynamic driving force for each step is determined by the **free energy of solvation** of each species.  The question "why does CO₂ prefer to remain as neutral CO₂ in pure water while bicarbonate is the dominant aqueous carbon species at physiological pH?" is answered by comparing:

| Species | Net charge | $\Delta G_{\text{hyd}}$ (experiment) |
|---------|-----------|--------------------------------------|
| CO₂ | 0 | −8.3 kcal/mol |
| HCO₃⁻ | −1 | −85 kcal/mol |

The enormous difference (~77 kcal/mol) arises from **electrostatic solvation**: polar water molecules orient strongly around the charged bicarbonate ion.

### 1.2 The Alchemical Strategy

Direct computation of absolute solvation free energies requires sampling the full configurational space of the solute + solvent system.  Instead we use an **alchemical** (unphysical) thermodynamic cycle:

```
CO₂ (gas)  ──────────────────────────────►  HCO₃⁻ (gas)
    │  ΔG_hyd(CO₂)                                │  ΔG_hyd(HCO₃⁻)
    ▼                                              ▼
CO₂ (aq)  ──── ΔG_mut(aq) = ? ──────────►  HCO₃⁻ (aq)
```

The **Hess's law** relation gives:

$$\Delta\Delta G_{\text{hyd}} = \Delta G_{\text{hyd}}(\text{HCO}_3^-) - \Delta G_{\text{hyd}}(\text{CO}_2)
= \Delta G_{\text{mut}}(\text{aq}) - \Delta G_{\text{mut}}(\text{gas})$$

We *alchemically mutate* CO₂ → HCO₃⁻ in both phases and subtract, cancelling errors in the force field.

### 1.3 Scope of This Tutorial

We **model** the key physics with a simple but realistic force-field representation:

- CO₂ is represented as a linear triatomic with partial charges $q_C = +0.70e$, $q_O = -0.35e$ and LJ parameters from the EPM2 model.
- HCO₃⁻ is represented with the same geometry plus one extra unit of negative charge distributed across the oxygens.
- Water is SPC/E.
- The alchemical mutation gradually **transfers charge** from CO₂ to HCO₃⁻ controlled by $\lambda \in [0,1]$.

All MD is performed by a minimal Langevin integrator written in NumPy so that no external program is needed.

---
## 2. Free Energy Perturbation (FEP) Theory

### 2.1 The Zwanzig Equation

Consider two Hamiltonians $H_0$ and $H_1$ differing by a perturbation $\Delta H = H_1 - H_0$.  The exact free energy difference is:

$$\Delta G = G_1 - G_0 = -k_B T \ln \left\langle e^{-\Delta H / k_B T} \right\rangle_0$$

This is the **Zwanzig relation** (1954).  The average is taken over configurations sampled from the **reference** ensemble at $H_0$.

**Proof:**

$$e^{-\beta G_1} = Z_1 = \int e^{-\beta H_1}\,d\mathbf{r} = \int e^{-\beta (H_0 + \Delta H)}\,d\mathbf{r}$$

$$= \int e^{-\beta H_0} e^{-\beta \Delta H}\,d\mathbf{r} = Z_0 \left\langle e^{-\beta \Delta H} \right\rangle_0$$

$$\Rightarrow G_1 - G_0 = -k_B T \ln \left\langle e^{-\beta \Delta H} \right\rangle_0$$

### 2.2 Staged FEP (λ-windows)

Zwanzig's equation converges poorly when $\Delta H$ is large.  The standard solution is to introduce a **coupling parameter** $\lambda \in [0,1]$ and stage the calculation:

$$H(\lambda) = (1-\lambda)\,H_0 + \lambda\,H_1$$

The total free energy difference is:

$$\Delta G = \sum_{i=1}^{N} \Delta G_{\lambda_i \to \lambda_{i+1}}, \quad \Delta G_{\lambda_i \to \lambda_{i+1}} = -k_BT\ln\left\langle e^{-\beta[H(\lambda_{i+1})-H(\lambda_i)]}\right\rangle_{\lambda_i}$$

### 2.3 Bennett Acceptance Ratio (BAR)

BAR uses **both** forward and backward samples to minimize variance.  At each window pair $(\lambda_i, \lambda_{i+1})$, the BAR estimator solves:

$$\sum_{j=1}^{n_0} \frac{1}{1+e^{\beta(\Delta H_{j}^{0\to1} - C)}} = \sum_{j=1}^{n_1} \frac{1}{1+e^{-\beta(\Delta H_{j}^{1\to0} + C)}}$$

for $C = \Delta G + k_BT\ln(n_1/n_0)$.  BAR achieves **minimum variance** for a given total simulation length.

### 2.4 Thermodynamic Integration (TI) — an alternative

Instead of exponential averaging, TI integrates the ensemble average of $\partial H/\partial\lambda$:

$$\Delta G = \int_0^1 \left\langle \frac{\partial H}{\partial\lambda} \right\rangle_\lambda d\lambda$$

In practice: run $N_\lambda$ simulations, collect $\langle \partial H/\partial\lambda \rangle_\lambda$ at each window, integrate numerically.

---
## 3. Setup and Imports

In [ ]:
# =============================================================================
# Ch121a: Molecular Dynamics — Notebook 07a: CO2 vs Bicarbonate FEP Tutorial
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import brentq
from scipy.integrate import simpson

rng = np.random.default_rng(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.labelsize': 12, 'legend.fontsize': 10})

# Physical constants
kB     = 0.001987204   # kcal/(mol·K)
T      = 298.15        # K  (25 °C)
beta   = 1.0 / (kB * T)

print(f"Temperature : {T} K")
print(f"kBT         : {1/beta:.4f} kcal/mol")

---
## 4. Molecular Models

### 4.1 Force-Field Parameters

We use a simplified but physically motivated model.  For the solvation problem the dominant contribution to $\Delta\Delta G_{\text{hyd}}$ is **electrostatic**, so we focus on the charge-change perturbation and treat the LJ core as constant.

**CO₂ (EPM2-like model)**

| Site | $q/e$ | $\sigma$ (Å) | $\varepsilon$ (kcal/mol) |
|------|-------|--------------|-------------------------|
| C    | +0.70 | 2.757 | 0.0559 |
| O₁   | −0.35 | 3.033 | 0.1597 |
| O₂   | −0.35 | 3.033 | 0.1597 |

**HCO₃⁻ (bicarbonate, net charge −1)**  
Same geometry; the extra charge (−1e) is distributed: $\Delta q_C = -0.30$, $\Delta q_O = -0.233$ each.

| Site | $q/e$ | $\sigma$ (Å) | $\varepsilon$ (kcal/mol) |
|------|-------|--------------|-------------------------|
| C    | +0.40 | 2.757 | 0.0559 |
| O₁   | −0.583 | 3.033 | 0.1597 |
| O₂   | −0.583 | 3.033 | 0.1597 |
| O₃ (OH) | −0.583 | 3.033 | 0.1597 |

For simplicity in the FEP demonstration, we model the solute as a **single interaction site** with an effective radius and a variable charge $q(\lambda)$.

In [ ]:
# Solute effective parameters (reduced single-site model for FEP demo)
# CO2:   neutral (q=0 in perturbation sense)
# HCO3-: charge -1e

q_CO2   =  0.00   # net charge in units of e (alchemical end-state 0)
q_HCO3  = -1.00   # net charge in units of e (alchemical end-state 1)

# Born model effective radii (Å)
R_CO2   =  2.87   # effective radius for CO2
R_HCO3  =  2.12   # effective radius for HCO3-

# Dielectric constant of water at 298 K
eps_w   = 78.4

# Unit conversion: 1 e² / Å → kcal/mol
# Coulomb's constant in kcal·Å/(mol·e²)
ke_kcal = 332.0637   # kcal·Å/(mol·e²)

print("Force-field parameters loaded.")
print(f"  CO2   effective radius : {R_CO2}  Å,  charge : {q_CO2} e")
print(f"  HCO3- effective radius : {R_HCO3} Å,  charge : {q_HCO3} e")

---
## 5. The Born Model of Solvation

### 5.1 Analytical Benchmark

Before running FEP, let us compute the analytical **Born solvation free energy** as a reference.

The Born model treats the solvent as a dielectric continuum and the solute as a sphere of radius $R$ with charge $q$:

$$\Delta G_{\text{Born}} = -\frac{k_e q^2}{2R}\left(1 - \frac{1}{\varepsilon}\right)$$

where $k_e = 332.06$ kcal·Å/(mol·e²), $R$ is in Å, $\varepsilon = 78.4$ for water.

### 5.2 Physical Interpretation

The Born model captures two key physics:
- **Quadratic in charge**: doubling the charge quadruples the solvation energy.
- **Inversely proportional to radius**: small ions are better solvated (tighter water shells).

The neutral CO₂ has $q=0$ so $\Delta G_{\text{Born}} = 0$ (no electrostatic solvation; only dispersion/cavity terms remain).

In [ ]:
def born_solvation(q, R, ke=ke_kcal, eps=eps_w):
    """Born model electrostatic solvation free energy (kcal/mol)."""
    return -(ke * q**2) / (2 * R) * (1 - 1/eps)

dG_born_CO2   = born_solvation(q_CO2,  R_CO2)
dG_born_HCO3  = born_solvation(q_HCO3, R_HCO3)
ddG_born      = dG_born_HCO3 - dG_born_CO2

print("Born model solvation free energies:")
print(f"  ΔG_hyd(CO₂)   = {dG_born_CO2:.2f} kcal/mol")
print(f"  ΔG_hyd(HCO₃⁻) = {dG_born_HCO3:.2f} kcal/mol")
print(f"  ΔΔG_hyd       = {ddG_born:.2f} kcal/mol")
print(f"\nExperimental ΔΔG_hyd ≈ −77 kcal/mol (Born underestimates due to discrete solvent effects)")

### 5.3 Born Energy as a Function of Charge and Radius

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: Born energy vs charge for fixed R
q_vals = np.linspace(-2, 2, 200)
for R, label, color in [(R_CO2, 'R = 2.87 Å (CO₂)', 'C0'),
                         (R_HCO3, 'R = 2.12 Å (HCO₃⁻)', 'C1'),
                         (1.5, 'R = 1.50 Å (small ion)', 'C2')]:
    axes[0].plot(q_vals, born_solvation(q_vals, R), label=label, color=color)
axes[0].axvline(0, color='gray', lw=0.8, ls='--')
axes[0].axvline(-1, color='C1', lw=0.8, ls=':')
axes[0].set_xlabel('Net charge $q$ (e)')
axes[0].set_ylabel('$\\Delta G_{\\rm Born}$ (kcal/mol)')
axes[0].set_title('Born Solvation Free Energy vs Charge')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: Born energy vs radius for q = -1
R_vals = np.linspace(1.0, 5.0, 200)
axes[1].plot(R_vals, born_solvation(-1.0, R_vals), 'C3', lw=2)
axes[1].axvline(R_CO2,  color='C0', ls='--', label=f'R(CO₂) = {R_CO2} Å')
axes[1].axvline(R_HCO3, color='C1', ls='--', label=f'R(HCO₃⁻) = {R_HCO3} Å')
axes[1].scatter([R_HCO3], [born_solvation(-1.0, R_HCO3)], color='C1', zorder=5, s=80)
axes[1].set_xlabel('Effective radius $R$ (Å)')
axes[1].set_ylabel('$\\Delta G_{\\rm Born}$ for $q=-1$ (kcal/mol)')
axes[1].set_title('Born Solvation Free Energy vs Radius ($q=-1$)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('born_model.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

---
## 6. Free Energy Perturbation Calculation

### 6.1 Hamiltonian and Coupling

We model the **electrostatic contribution** to solvation using a simple ion-dipole model.  The interaction energy between the alchemical solute (charge $q(\lambda)$) and a single water molecule modelled as a dipole $\boldsymbol{\mu}$ at distance $r$ is:

$$U_{\text{elec}}(r, \theta; \lambda) = \frac{k_e\, q(\lambda)\, |\boldsymbol{\mu}|\cos\theta}{r^2}$$

where $\theta$ is the angle between the dipole and the solute–water vector.

The linear coupling:

$$q(\lambda) = (1-\lambda)\,q_0 + \lambda\,q_1 = -\lambda \quad [e]$$

We represent the first solvation shell as $N_w$ water molecules.  Each water's orientation $\theta$ fluctuates thermally.  The probability distribution $P(\theta)$ follows Boltzmann statistics:

$$P(\theta | r, \lambda) \propto \exp\!\left(-\beta U_{\text{elec}}(r,\theta;\lambda)\right)\sin\theta$$

This is the **Langevin dipole model** for orientational polarization.

### 6.2 Simulation Strategy

For each $\lambda$-window we:
1. Generate $N_{\text{MC}}$ Monte-Carlo configurations by sampling $r$ and $\theta$ for each water molecule.
2. Evaluate $\Delta U = U(\lambda+\delta\lambda) - U(\lambda)$ for every configuration.
3. Compute the FEP estimate $\Delta G_{\lambda}$ using the Zwanzig formula.
4. Also compute the BAR estimate using forward *and* backward samples.
5. Use TI by collecting $\langle \partial U/\partial\lambda \rangle_\lambda$.

In [ ]:
# =============================================================================
# Solvation Shell Model
# =============================================================================

# Water parameters (SPC/E)
mu_water  = 2.35   # Debye  (dipole moment)
# Convert to kcal/mol·Å/e:  1 D = 0.20819 e·Å
mu_water_eA = mu_water * 0.20819   # e·Å

# Solvation shell geometry
N_shell   = 8      # number of water molecules in the first shell
r_shell   = 3.5    # Å  (O···solute centre-of-mass distance)

def U_single_water(q_lam, r, theta):
    """
    Ion-dipole interaction energy (kcal/mol).
    q_lam : solute charge (e)
    r     : solute-water distance (Å)
    theta : angle between dipole and solute-water vector (rad)
    """
    return -ke_kcal * q_lam * mu_water_eA * np.cos(theta) / r**2

def U_total(q_lam, r_arr, theta_arr):
    """Total ion-dipole energy summed over all shell waters."""
    return np.sum(U_single_water(q_lam, r_arr, theta_arr))

print(f"Water dipole moment : {mu_water} D = {mu_water_eA:.4f} e·Å")
print(f"Shell geometry      : {N_shell} waters at r = {r_shell} Å")

In [ ]:
# =============================================================================
# Monte Carlo Sampling for FEP
# =============================================================================

N_lambda = 20       # number of λ-windows
N_MC     = 5000     # MC steps per window (increased for convergence)
dlambda  = 1.0 / N_lambda

lambdas = np.linspace(0.0, 1.0, N_lambda + 1)  # λ = 0 (CO2) → 1 (HCO3-)

def sample_configurations(q_lam, N_samples, N_w=N_shell, r=r_shell, rng=rng):
    """
    Metropolis MC for water orientations around solute with charge q_lam.
    Returns array of total energies and (r_arr, theta_arr) configurations.
    """
    # Start from random orientations
    theta = np.arccos(rng.uniform(-1, 1, N_w))  # uniformly random on sphere
    r_arr = np.full(N_w, r)

    configs   = []
    energies  = []
    U_curr    = U_total(q_lam, r_arr, theta)

    accepted  = 0
    delta_max = np.pi / 4   # max trial move (rad)

    for step in range(N_samples):
        # Propose: perturb one random water's theta
        idx        = rng.integers(0, N_w)
        theta_old  = theta[idx]
        theta_new  = theta_old + rng.uniform(-delta_max, delta_max)
        theta_new  = np.clip(theta_new, 0, np.pi)

        # Energy change
        dU = (U_single_water(q_lam, r, theta_new)
            - U_single_water(q_lam, r, theta_old))

        # Metropolis criterion with sin(theta) Jacobian
        log_acc = -beta * dU + np.log(
            np.sin(theta_new) / np.sin(theta_old + 1e-12))

        if np.log(rng.random() + 1e-300) < log_acc:
            theta[idx] = theta_new
            U_curr    += dU
            accepted  += 1

        configs.append(theta.copy())
        energies.append(U_curr)

    return np.array(configs), np.array(energies), r_arr.copy(), accepted / N_samples

print("MC sampler defined.")

In [ ]:
# =============================================================================
# Run FEP over all λ-windows
# =============================================================================
print("Running FEP windows...")

# Storage
dG_FEP_fwd  = np.zeros(N_lambda)   # forward FEP  λ_i → λ_{i+1}
dG_FEP_bwd  = np.zeros(N_lambda)   # backward FEP λ_{i+1} → λ_i
dHdl_mean   = np.zeros(N_lambda)   # TI: ⟨∂H/∂λ⟩
dHdl_std    = np.zeros(N_lambda)   # TI: standard error
accept_rate = np.zeros(N_lambda)

# Store all configurations for BAR
all_configs = {}   # lambda_i : (theta_configs, r_arr)

for i in range(N_lambda):
    lam_i   = lambdas[i]
    lam_ip1 = lambdas[i + 1]
    q_i     = q_CO2 + lam_i   * (q_HCO3 - q_CO2)    # charge at λ_i
    q_ip1   = q_CO2 + lam_ip1 * (q_HCO3 - q_CO2)    # charge at λ_{i+1}

    # Sample at λ_i
    configs_i, E_i, r_arr, acc = sample_configurations(q_i, N_MC)
    accept_rate[i] = acc
    all_configs[i] = (configs_i, r_arr)

    # Forward FEP: ΔG(λ_i → λ_{i+1})
    dH_fwd = np.array([
        U_total(q_ip1, r_arr, th) - U_total(q_i, r_arr, th)
        for th in configs_i
    ])
    dG_FEP_fwd[i] = -(1.0/beta) * np.log(np.mean(np.exp(-beta * dH_fwd)))

    # TI: ∂H/∂λ = (H₁ - H₀) for linear coupling
    dU_dl = np.array([
        U_total(q_HCO3 - q_CO2, r_arr, th)   # dH/dλ = (H1-H0) for linear coupling
        for th in configs_i
    ])
    dHdl_mean[i] = np.mean(dU_dl)
    dHdl_std[i]  = np.std(dU_dl) / np.sqrt(len(dU_dl))

print(f"Done.  Mean acceptance rate: {accept_rate.mean():.2%}")

In [ ]:
# =============================================================================
# Backward FEP & BAR
# =============================================================================
print("Running backward windows for BAR...")

dG_BAR = np.zeros(N_lambda)

for i in range(N_lambda):
    lam_i   = lambdas[i]
    lam_ip1 = lambdas[i + 1]
    q_i     = q_CO2 + lam_i   * (q_HCO3 - q_CO2)
    q_ip1   = q_CO2 + lam_ip1 * (q_HCO3 - q_CO2)

    # Backward: sample at λ_{i+1}, measure ΔH going back to λ_i
    configs_ip1, _, r_arr_ip1, _ = sample_configurations(q_ip1, N_MC)

    # Forward ΔH values (sampled at λ_i)
    configs_i, r_arr_i = all_configs[i]
    dH_fwd = np.array([
        U_total(q_ip1, r_arr_i, th) - U_total(q_i, r_arr_i, th)
        for th in configs_i
    ])
    # Backward ΔH values (sampled at λ_{i+1})
    dH_bwd = np.array([
        U_total(q_i, r_arr_ip1, th) - U_total(q_ip1, r_arr_ip1, th)
        for th in configs_ip1
    ])

    # BAR: solve for C iteratively
    n0 = len(dH_fwd)
    n1 = len(dH_bwd)

    def bar_residual(C):
        lhs = np.sum(1.0 / (1.0 + np.exp( beta * (dH_fwd - C))))
        rhs = np.sum(1.0 / (1.0 + np.exp(-beta * (dH_bwd + C))))
        return lhs - rhs

    try:
        C_opt = brentq(bar_residual, -20.0, 20.0, xtol=1e-8)
        dG_BAR[i] = C_opt - (1.0/beta) * np.log(n1 / n0)
    except ValueError:
        # Fallback to forward FEP if BAR fails to converge
        dG_BAR[i] = dG_FEP_fwd[i]

print("BAR calculation complete.")

In [ ]:
# =============================================================================
# Accumulate total ΔG
# =============================================================================
lam_mid = 0.5 * (lambdas[:-1] + lambdas[1:])

dG_total_FEP = np.sum(dG_FEP_fwd)
dG_total_BAR = np.sum(dG_BAR)
dG_total_TI  = simpson(dHdl_mean, x=lam_mid)

print("=" * 55)
print("  Alchemical mutation: CO₂ → HCO₃⁻  (in aqueous shell)")
print("=" * 55)
print(f"  FEP (forward Zwanzig) :  {dG_total_FEP:+.2f}  kcal/mol")
print(f"  BAR                  :  {dG_total_BAR:+.2f}  kcal/mol")
print(f"  TI                   :  {dG_total_TI:+.2f}  kcal/mol")
print(f"  Born model reference :  {ddG_born:+.2f}  kcal/mol")
print("=" * 55)
print("Note: The shell model gives the first-shell contribution only.")
print("Full solvation requires outer shells + LJ + cavity terms.")

---
## 7. Analysis and Convergence Diagnostics

### 7.1 λ-dependent Free Energy Profile

The cumulative free energy as a function of $\lambda$ reveals:
- Whether the change is gradual or dominated by a few windows.
- Whether more $\lambda$-windows are needed in certain regions.

In [ ]:
cumulative_FEP = np.concatenate([[0], np.cumsum(dG_FEP_fwd)])
cumulative_BAR = np.concatenate([[0], np.cumsum(dG_BAR)])
cumulative_TI  = np.concatenate([[0], np.cumsum(
    dHdl_mean * dlambda)])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) Cumulative free energy profile
ax = axes[0]
ax.plot(lambdas, cumulative_FEP, 'o-', label='FEP (Zwanzig)', color='C0')
ax.plot(lambdas, cumulative_BAR, 's-', label='BAR', color='C1')
ax.plot(lambdas, cumulative_TI,  '^-', label='TI',  color='C2')
ax.set_xlabel('$\\lambda$')
ax.set_ylabel('$\\Delta G(\\lambda)$  (kcal/mol)')
ax.set_title('Cumulative Free Energy Profile')
ax.legend()
ax.grid(alpha=0.3)

# (b) Per-window free energy increments
ax = axes[1]
ax.bar(lam_mid, dG_FEP_fwd, width=dlambda*0.3, alpha=0.7, label='FEP', color='C0')
ax.bar(lam_mid + dlambda*0.3, dG_BAR, width=dlambda*0.3, alpha=0.7, label='BAR', color='C1')
ax.set_xlabel('$\\lambda$ window midpoint')
ax.set_ylabel('$\\delta G$ (kcal/mol)')
ax.set_title('Per-window Free Energy Increments')
ax.legend()
ax.grid(alpha=0.3)

# (c) TI integrand ⟨∂H/∂λ⟩
ax = axes[2]
ax.errorbar(lam_mid, dHdl_mean, yerr=dHdl_std * 2, fmt='o-', color='C3',
            capsize=4, label='$\\langle \\partial H/\\partial\\lambda \\rangle \\pm 2\\sigma$')
ax.fill_between(lam_mid, dHdl_mean - 2*dHdl_std, dHdl_mean + 2*dHdl_std,
                alpha=0.3, color='C3')
ax.set_xlabel('$\\lambda$')
ax.set_ylabel('$\\langle \\partial H / \\partial \\lambda \\rangle$ (kcal/mol)')
ax.set_title('TI Integrand')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fep_profile.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

### 7.2 Phase Space Overlap

FEP convergence requires **phase space overlap** between adjacent $\lambda$-windows.  A standard diagnostic is the **overlap matrix**: the fraction of configurations from window $i$ that have non-negligible weight in window $j$.

The overlap integral for two windows is:

$$O_{ij} = \int \frac{P_i(x)\, P_j(x)}{P_i(x)+P_j(x)}\,dx$$

In practice we estimate it via the **Bhattacharyya coefficient** on the energy distributions.

In [ ]:
# Energy distribution overlap between adjacent windows
# Quick proxy: overlap of ΔH histograms between windows i and i+1

def bhattacharyya_overlap(a, b, n_bins=50):
    """Bhattacharyya coefficient for overlap of two distributions."""
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    bins = np.linspace(lo, hi, n_bins + 1)
    ha, _ = np.histogram(a, bins=bins, density=True)
    hb, _ = np.histogram(b, bins=bins, density=True)
    db = bins[1] - bins[0]
    return np.sum(np.sqrt(ha * hb)) * db

overlaps = []
for i in range(N_lambda - 1):
    lam_i   = lambdas[i]
    lam_ip1 = lambdas[i + 1]
    lam_ip2 = lambdas[i + 2]
    q_i     = q_CO2 + lam_i   * (q_HCO3 - q_CO2)
    q_ip1   = q_CO2 + lam_ip1 * (q_HCO3 - q_CO2)
    q_ip2   = q_CO2 + lam_ip2 * (q_HCO3 - q_CO2)

    configs_i,   r_arr_i   = all_configs[i]
    configs_ip1, r_arr_ip1 = all_configs[i + 1]

    E_i    = np.array([U_total(q_ip1, r_arr_i,   th) - U_total(q_i,   r_arr_i,   th) for th in configs_i])
    E_ip1  = np.array([U_total(q_ip2, r_arr_ip1, th) - U_total(q_ip1, r_arr_ip1, th) for th in configs_ip1])

    overlaps.append(bhattacharyya_overlap(E_i, E_ip1))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(range(len(overlaps)), overlaps, color='steelblue', alpha=0.8)
ax.axhline(0.5, color='red', ls='--', label='0.5 threshold')
ax.set_xlabel('Window pair index')
ax.set_ylabel('Bhattacharyya overlap coefficient')
ax.set_title('Phase-Space Overlap Between Adjacent λ-Windows')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fep_overlap.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Mean overlap: {np.mean(overlaps):.3f}  (>0.5 is good)")

### 7.3 Convergence with Sample Size

In [ ]:
# Test convergence of FEP estimate at the midpoint window (λ=0.5)
lam_test = 0.5
q_test   = q_CO2 + lam_test * (q_HCO3 - q_CO2)
q_next   = q_CO2 + (lam_test + dlambda) * (q_HCO3 - q_CO2)

sample_sizes = [50, 100, 200, 500, 1000, 2000, 5000, 10000]
dG_fep_convergence = []

for N in sample_sizes:
    configs, _, r_arr, _ = sample_configurations(q_test, N, rng=rng)
    dH = np.array([U_total(q_next, r_arr, th) - U_total(q_test, r_arr, th)
                   for th in configs])
    dG_fep_convergence.append(-(1.0/beta) * np.log(np.mean(np.exp(-beta * dH))))

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(sample_sizes, dG_fep_convergence, 'o-', color='C4')
ax.axhline(dG_fep_convergence[-1], color='gray', ls='--', label='Converged value')
ax.set_xlabel('Number of MC samples')
ax.set_ylabel('$\\Delta G_{\\lambda=0.5}$ (kcal/mol)')
ax.set_title('FEP Convergence with Sample Size (λ = 0.5 window)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fep_convergence.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8. Thermodynamic Cycle and ΔΔG_hyd

### 8.1 Cycle Diagram

The full thermodynamic cycle is:

```
CO₂(g) ────── ΔG_mut(gas) ──────► HCO₃⁻(g)
  │                                    │
  │ ΔG_hyd(CO₂)             ΔG_hyd(HCO₃⁻) │
  │                                    │
  ▼                                    ▼
CO₂(aq) ──── ΔG_mut(aq) ────────► HCO₃⁻(aq)
```

Hess's law:
$$\Delta\Delta G_{\text{hyd}} = \Delta G_{\text{mut}}(\text{aq}) - \Delta G_{\text{mut}}(\text{gas})$$

In the gas phase (vacuum), the mutation energy is purely electrostatic self-energy: the **Born charging energy** in vacuum.

### 8.2 Gas-Phase Correction

In vacuum ($\varepsilon=1$), the charging free energy of a sphere from $q_0$ to $q_1$ is:

$$\Delta G_{\text{mut}}(\text{gas}) = \int_{q_0}^{q_1} \frac{k_e q}{R}\,dq = \frac{k_e}{2R}(q_1^2 - q_0^2)$$

For CO₂($q=0$) → HCO₃⁻($q=-1$):

$$\Delta G_{\text{mut}}(\text{gas}) = \frac{k_e}{2R_{\text{eff}}}((-1)^2 - 0^2) = \frac{k_e}{2 R_{\text{eff}}}$$

In [ ]:
# Gas-phase mutation: charging energy in vacuum (ε=1)
R_eff_gas = 0.5 * (R_CO2 + R_HCO3)   # effective radius for the mutation path

dG_mut_gas = (ke_kcal / (2 * R_eff_gas)) * (q_HCO3**2 - q_CO2**2)
dG_mut_aq  = dG_total_BAR   # our best FEP estimate from the shell model

# Correct for outer-shell and continuum contributions using Born model
dG_born_aq  = born_solvation(q_HCO3, R_HCO3) - born_solvation(q_CO2, R_CO2)
dG_born_gas = (ke_kcal / (2 * R_HCO3)) * q_HCO3**2 - (ke_kcal / (2 * R_CO2)) * q_CO2**2

ddG_FEP_shell = dG_mut_aq - dG_mut_gas
ddG_Born_full = dG_born_aq - dG_born_gas   # full Born thermodynamic cycle

print("Thermodynamic Cycle Summary")
print("-" * 55)
print(f"  ΔG_mut (aq,  shell model) = {dG_mut_aq:+.2f} kcal/mol")
print(f"  ΔG_mut (gas, charging)    = {dG_mut_gas:+.2f} kcal/mol")
print(f"  ΔΔG_hyd (shell FEP)       = {ddG_FEP_shell:+.2f} kcal/mol")
print()
print(f"  ΔΔG_hyd (full Born model) = {ddG_Born_full:+.2f} kcal/mol")
print(f"  Experimental ΔΔG_hyd      ≈ −77 kcal/mol")
print()
print("The shell model captures ~1st-shell electrostatics.")
print("Full agreement requires continuum outer shells + LJ + cavity.")

---
## 9. Detailed Analysis: Charge–Water Interaction

### 9.1 Orientational Ordering of Water

As the solute gains charge, water molecules become **orientationally ordered**.  We can visualize this by looking at the distribution of $\cos\theta$ at different $\lambda$ values.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# (a) cos(theta) distributions at selected lambda values
ax = axes[0]
lambda_sample = [0.0, 0.25, 0.5, 0.75, 1.0]
colors = plt.cm.viridis(np.linspace(0, 1, len(lambda_sample)))

cos_means = []
for lam, col in zip(lambda_sample, colors):
    q = q_CO2 + lam * (q_HCO3 - q_CO2)
    configs, _, _, _ = sample_configurations(q, 3000, rng=rng)
    cos_theta = np.cos(configs).flatten()
    ax.hist(cos_theta, bins=40, density=True, alpha=0.6, color=col,
            label=f'λ={lam:.2f}')
    cos_means.append(np.mean(cos_theta))

ax.set_xlabel('$\\cos\\theta$ (water orientation)')
ax.set_ylabel('Probability density')
ax.set_title('Water Orientational Distributions')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# (b) Mean cos(theta) vs lambda → order parameter
ax = axes[1]
ax.plot(lambda_sample, cos_means, 'o-', color='darkgreen', lw=2)
ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.set_xlabel('$\\lambda$  (0=CO₂, 1=HCO₃⁻)')
ax.set_ylabel('$\\langle \\cos\\theta \\rangle$  (order parameter)')
ax.set_title('Orientational Order of Water Shell')
ax.grid(alpha=0.3)
ax.annotate('Random (neutral)', xy=(0, cos_means[0]), fontsize=9,
            xytext=(0.1, cos_means[0] + 0.05))
ax.annotate('Strongly ordered\n(charged)', xy=(1, cos_means[-1]), fontsize=9,
            xytext=(0.6, cos_means[-1] + 0.05))

plt.tight_layout()
plt.savefig('water_orientation.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

---
## 10. Choosing the λ-Schedule

### 10.1 Effect of Window Spacing

Not all $\lambda$-spacings give equally good convergence.  Non-uniform spacings can reduce total variance:

- **Uniform**: $\lambda = 0, 1/N, 2/N, \ldots, 1$
- **Gauss–Legendre quadrature**: optimal for TI if integrand is smooth
- **Soft-core potential**: prevents divergence when creating/annihilating charges or LJ sites

The **optimal** schedule concentrates windows where $\langle(\partial H/\partial\lambda)^2\rangle - \langle\partial H/\partial\lambda\rangle^2$ is largest (i.e., where variance is highest).

In [ ]:
# Gauss-Legendre quadrature points vs uniform for TI
n_gl = N_lambda
gl_points, gl_weights = np.polynomial.legendre.leggauss(n_gl)
lam_GL = 0.5 * (gl_points + 1)   # map from [-1,1] to [0,1]
w_GL   = 0.5 * gl_weights

# Evaluate TI integrand at GL points
dHdl_GL = []
for lam in lam_GL:
    q = q_CO2 + lam * (q_HCO3 - q_CO2)
    configs, _, r_arr, _ = sample_configurations(q, N_MC, rng=rng)
    dU = np.array([U_total(q_HCO3 - q_CO2, r_arr, th) for th in configs])
    dHdl_GL.append(np.mean(dU))

dHdl_GL = np.array(dHdl_GL)
dG_TI_GL = np.dot(w_GL, dHdl_GL)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Integrand comparison
ax = axes[0]
ax.plot(lam_mid,  dHdl_mean, 'o-', label=f'Uniform  ΔG={dG_total_TI:.2f}', color='C0')
ax.plot(lam_GL,   dHdl_GL,   's-', label=f'GL quad  ΔG={dG_TI_GL:.2f}',   color='C1')
ax.set_xlabel('$\\lambda$')
ax.set_ylabel('$\\langle \\partial H/\\partial\\lambda \\rangle$')
ax.set_title('TI Integrand: Uniform vs Gauss–Legendre')
ax.legend()
ax.grid(alpha=0.3)

# Window positions
ax = axes[1]
ax.scatter(lam_mid, np.zeros_like(lam_mid),
           s=60, label='Uniform windows', color='C0', marker='|')
ax.scatter(lam_GL, np.ones_like(lam_GL),
           s=60, label='Gauss–Legendre points', color='C1', marker='|')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Uniform', 'GL'])
ax.set_xlabel('$\\lambda$')
ax.set_title('λ-Window Positions')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('lambda_schedule.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"TI (uniform):         {dG_total_TI:.3f} kcal/mol")
print(f"TI (Gauss–Legendre):  {dG_TI_GL:.3f} kcal/mol")

---
## 11. Statistical Error Analysis

### 11.1 Bootstrap Confidence Intervals

The uncertainty in $\Delta G$ from FEP has two contributions:
1. **Statistical error**: finite sampling → use bootstrap or block averaging
2. **Systematic error**: insufficient overlap, hysteresis, force-field inaccuracy

We use **bootstrap resampling** to estimate the 95% confidence interval on $\Delta G_{\text{total}}$.

In [ ]:
# Bootstrap error analysis on the midpoint window
lam_test  = 0.5
q_test    = q_CO2 + lam_test * (q_HCO3 - q_CO2)
q_next    = q_CO2 + (lam_test + dlambda) * (q_HCO3 - q_CO2)

configs_bs, _, r_arr_bs, _ = sample_configurations(q_test, 5000, rng=rng)
dH_bs = np.array([
    U_total(q_next, r_arr_bs, th) - U_total(q_test, r_arr_bs, th)
    for th in configs_bs
])

N_boot = 2000
dG_boot = np.zeros(N_boot)
n_samp  = len(dH_bs)
for b in range(N_boot):
    idx  = rng.integers(0, n_samp, n_samp)
    dH_b = dH_bs[idx]
    dG_boot[b] = -(1/beta) * np.log(np.mean(np.exp(-beta * dH_b)))

ci95 = np.percentile(dG_boot, [2.5, 97.5])
dG_mean_boot = np.mean(dG_boot)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(dG_boot, bins=60, density=True, color='steelblue', alpha=0.75, edgecolor='white')
ax.axvline(dG_mean_boot, color='red',   lw=2, label=f'Mean = {dG_mean_boot:.3f}')
ax.axvline(ci95[0],      color='orange', lw=1.5, ls='--', label=f'2.5%  = {ci95[0]:.3f}')
ax.axvline(ci95[1],      color='orange', lw=1.5, ls='--', label=f'97.5% = {ci95[1]:.3f}')
ax.set_xlabel('$\\Delta G_{\\lambda=0.5}$ (kcal/mol)')
ax.set_ylabel('Probability density')
ax.set_title('Bootstrap Distribution of FEP Free Energy Estimate')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fep_bootstrap.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"FEP at λ=0.5:  {dG_mean_boot:.3f} [{ci95[0]:.3f}, {ci95[1]:.3f}] kcal/mol (95% CI)")

---
## 12. Summary and Discussion

### 12.1 Key Results

In [ ]:
print("=" * 60)
print("  SUMMARY: CO₂ vs HCO₃⁻ Solvation Free Energy")
print("=" * 60)
print()
print("  Method            ΔG_mut(aq)  [kcal/mol]")
print("  ─" * 30)
print(f"  FEP (Zwanzig)     {dG_total_FEP:+.2f}")
print(f"  BAR               {dG_total_BAR:+.2f}")
print(f"  TI (uniform)      {dG_total_TI:+.2f}")
print(f"  TI (GL quad.)     {dG_TI_GL:+.2f}")
print()
print("  Born model (analytical reference)")
print("  ─" * 30)
print(f"  ΔG_hyd(CO₂)       {dG_born_CO2:+.2f}")
print(f"  ΔG_hyd(HCO₃⁻)     {dG_born_HCO3:+.2f}")
print(f"  ΔΔG_hyd (Born)    {ddG_born:+.2f}")
print()
print("  Experimental (literature)")
print("  ─" * 30)
print("  ΔG_hyd(CO₂)       −8.3   kcal/mol")
print("  ΔG_hyd(HCO₃⁻)     −85    kcal/mol")
print("  ΔΔG_hyd            ≈ −77  kcal/mol")
print("=" * 60)

### 12.2 Physical Interpretation

**Why does bicarbonate have such a large (negative) solvation free energy?**

1. **Charge–dipole interaction**: a −1e charge at ~2 Å from water oxygens creates strong stabilising interactions (~−10 to −15 kcal/mol per water molecule × 6–8 waters in the first shell).

2. **Charge-induced dipole**: the charge polarises water beyond the permanent dipole, adding another ~5–10%.

3. **Born (continuum) contribution**: beyond the first shell, the dielectric response of the bulk water adds a large contribution captured by the Born model.

**Why is this important for CO₂ capture and ocean acidification?**

- At low pH (ocean surface), proton activity is high → CO₂(aq) is the stable form.
- At physiological pH ≈ 7.4, the $\text{pK}_a$ of carbonic acid means bicarbonate dominates.
- The large $\Delta\Delta G_{\text{hyd}}$ explains why deprotonation is thermodynamically favourable in water despite the apparent simplicity of the reaction.

### 12.3 Limitations of This Tutorial Model

| Simplification | Impact | How to fix |
|---|---|---|
| Single-site solute | Misses intramolecular charge distribution | Use all-atom CO₂/HCO₃⁻ | 
| Shell model (MC) | Ignores outer-shell and bulk contributions | Run full MD in periodic box |
| No LJ change | Ignores cavity formation and dispersion change | Add soft-core LJ perturbation |
| SPC/E water as background | SPC/E polarisability not included | Use polarisable FF (AMOEBA, Drude) |
| Classical force field | Misses charge transfer, QM effects | Use QM/MM FEP |

### 12.4 Comparison of FEP Estimators

| Estimator | Formula | Pros | Cons |
|-----------|---------|------|------|
| Zwanzig (FEP) | $-k_BT\ln\langle e^{-\beta\Delta H}\rangle_0$ | Simple | High variance; exponential average dominated by rare configs |
| BAR | Solve $\sum f(\Delta H_{01}-C) = \sum f(-\Delta H_{10}-C)$ | Minimum variance; uses both directions | Requires bidirectional sampling |
| TI | $\int_0^1 \langle \partial H/\partial\lambda\rangle_\lambda d\lambda$ | Numerically stable; no exponential | Requires smooth integrand; many windows |
| MBAR | Multistate generalisation of BAR | Uses all windows simultaneously | More complex to implement |

---
## 13. Exercises

1. **Effect of radius**: modify `R_HCO3` from 1.5 Å to 3.5 Å and plot how the Born solvation energy changes.  At what radius does the first-shell FEP calculation agree best with the Born model?

2. **Soft-core potentials**: in a real FEP calculation, creating a charge from zero can cause numerical instabilities when $q\to0$.  Implement a soft-core electrostatic potential of the form $U_{\text{sc}} = q(\lambda) / (r^2 + \alpha(1-\lambda))$ and compare its convergence to the standard linear coupling.

3. **BAR vs FEP asymmetry**: run the forward and backward FEP over all windows and compute the **hysteresis** $\Delta G_{\text{fwd}} + \Delta G_{\text{bwd}}$ (which should be zero).  How does it scale with $\delta\lambda$?  Plot the hysteresis vs $N_{\lambda}$.

4. **Number of shell waters**: vary `N_shell` from 1 to 16.  How does the total $\Delta G$ scale?  Compare to the continuum Born result.

5. **Dielectric environment**: repeat the Born model calculation for different solvents ($\varepsilon = 2$ for hexane, $\varepsilon = 36$ for acetonitrile, $\varepsilon = 78$ for water) and plot $\Delta G_{\text{Born}}$ vs $\varepsilon$.  Why do ionic species prefer polar solvents?

6. **MBAR implementation**: generalise the BAR code to use all $N_\lambda$ windows simultaneously (multistate BAR / MBAR).  Compare $\Delta G$ and its uncertainty to the pairwise BAR result.

---
## 14. Further Reading

| Reference | Topic |
|-----------|-------|
| Zwanzig, R. W. (1954) *J. Chem. Phys.* **22**, 1420 | Original FEP derivation |
| Bennett, C. H. (1976) *J. Comput. Phys.* **22**, 245 | BAR method |
| Shirts, M. R. & Chodera, J. D. (2008) *J. Chem. Phys.* **129**, 124105 | MBAR |
| Born, M. (1920) *Z. Phys.* **1**, 45 | Continuum solvation model |
| Shirts, M. R. *et al.* (2003) *J. Chem. Phys.* **119**, 5740 | Hydration free energies |
| Klimovich, P. V. *et al.* (2015) *J. Comput.-Aided Mol. Des.* **29**, 397 | Best practices for FEP |
| Straatsma, T. P. & Berendsen, H. J. C. (1988) *J. Chem. Phys.* **89**, 5876 | TI for solvation |
| Simonson, T. (2013) *Rep. Prog. Phys.* **66**, 737 | Electrostatics in biomolecular simulation |

In [ ]:
# Final comprehensive figure
fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig)

# Panel A: Thermodynamic cycle
ax0 = fig.add_subplot(gs[0])
labels = ['CO₂(g)', 'HCO₃⁻(g)', 'CO₂(aq)', 'HCO₃⁻(aq)']
x      = [0, 1, 0, 1]
y      = [1, 1, 0, 0]
ax0.scatter(x, y, s=200, zorder=5, color=['C0', 'C1', 'C0', 'C1'])
for xi, yi, lab in zip(x, y, labels):
    ax0.annotate(lab, (xi, yi), textcoords='offset points', xytext=(5, 5), fontsize=9)
# Arrows
for (x1,y1,x2,y2,label,col) in [
    (0,1,1,1,f'ΔG_mut(g)\n{dG_mut_gas:+.1f}','gray'),
    (0,0,1,0,f'ΔG_mut(aq)\n{dG_mut_aq:+.1f}','navy'),
    (0,1,0,0,f'ΔG_hyd\n(CO₂)','C0'),
    (1,1,1,0,f'ΔG_hyd\n(HCO₃⁻)','C1'),
]:
    ax0.annotate('', xy=(x2,y2), xytext=(x1,y1),
                 arrowprops=dict(arrowstyle='->', color=col, lw=1.5))
    ax0.text((x1+x2)/2, (y1+y2)/2, label, ha='center', va='center',
             fontsize=7.5, color=col)
ax0.set_xlim(-0.4, 1.5)
ax0.set_ylim(-0.4, 1.5)
ax0.axis('off')
ax0.set_title('Thermodynamic Cycle', fontsize=11)

# Panel B: Cumulative ΔG
ax1 = fig.add_subplot(gs[1])
ax1.plot(lambdas, cumulative_FEP, 'o-', lw=2, label='FEP', color='C0')
ax1.plot(lambdas, cumulative_BAR, 's-', lw=2, label='BAR', color='C1')
ax1.plot(lambdas, cumulative_TI,  '^-', lw=2, label='TI',  color='C2')
ax1.set_xlabel('$\\lambda$')
ax1.set_ylabel('$\\Delta G(\\lambda)$ (kcal/mol)')
ax1.set_title('Free Energy Profile')
ax1.legend()
ax1.grid(alpha=0.3)

# Panel C: Method comparison bar chart
ax2 = fig.add_subplot(gs[2])
methods = ['FEP\n(Zwanzig)', 'BAR', 'TI\n(uniform)', 'TI\n(GL)', 'Born\n(model)']
values  = [dG_total_FEP, dG_total_BAR, dG_total_TI, dG_TI_GL, ddG_Born_full]
colors_bar = ['C0', 'C1', 'C2', 'C3', 'C4']
bars = ax2.bar(methods, values, color=colors_bar, alpha=0.8)
ax2.axhline(0, color='k', lw=0.5)
ax2.set_ylabel('$\\Delta\\Delta G_{\\rm hyd}$ (kcal/mol)')
ax2.set_title('Method Comparison')
ax2.grid(alpha=0.3, axis='y')
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val:.1f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('fep_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print("Summary figure saved.")